In [2]:
import os
import pandas as pd

from finrl.meta.env_stock_trading.env_stocktrading import StockTradingEnv
from finrl.agents.stablebaselines3.models import DRLAgent
from stable_baselines3.common.logger import configure
from finrl import config_tickers
from finrl.main import check_and_make_directories
from finrl.config import INDICATORS, TRAINED_MODEL_DIR, RESULTS_DIR

check_and_make_directories([TRAINED_MODEL_DIR])

In [3]:
#Load Data from 01_data_ops_notebook
train = pd.read_csv('../data/train.csv')
date = train.columns[0]
train.index = pd.factorize(train[date])[0]
train.index.names = ['']
train.head()


,date,tic,close,high,low,open,volume,day,macd,boll_ub,boll_lb,rsi_30,cci_30,dx_30,close_30_sma,close_60_sma,vix,turbulence
,,,,,,,,,,,,,,,,,,
0,2009-01-02,A,10.259194,10.297098,9.798037,9.854891,4236220.0,4.0,0.0,10.924279,9.941555,100.0,66.666667,100.0,10.259194,10.259194,39.189999,0.0
0,2009-01-02,AAPL,2.714299,2.722974,2.547105,2.568640,746015200.0,4.0,0.0,10.924279,9.941555,100.0,66.666667,100.0,2.714299,2.714299,39.189999,0.0
0,2009-01-02,ABT,17.199024,17.292149,16.816896,17.189391,13163193.0,4.0,0.0,10.924279,9.941555,100.0,66.666667,100.0,17.199024,17.199024,39.189999,0.0
0,2009-01-02,ACGL,7.287052,7.395878,7.226829,7.395878,3445200.0,4.0,0.0,10.924279,9.941555,100.0,66.666667,100.0,7.287052,7.287052,39.189999,0.0
0,2009-01-02,ACN,24.225481,24.319016,23.412451,23.412451,3787400.0,4.0,0.0,10.924279,9.941555,100.0,66.666667,100.0,24.225481,24.225481,39.189999,0.0


In [4]:
#Setup Environment
stock_dimensions = len(train.tic.unique())
state_space = 1 + 2 * stock_dimensions + len(INDICATORS)*stock_dimensions
print(f"Stock Dimensions:  {stock_dimensions}, State Space: {state_space}")

Stock Dimensions:  414, State Space: 4141


In [5]:
buy_cost_list = sell_cost_list = [0.001] * stock_dimensions
num_stock_shares = [0] * stock_dimensions

env_tweeks = {
    "hmax": 100,
    "initial_amount": 1000,
    "num_stock_shares": num_stock_shares,
    "buy_cost_pct": buy_cost_list,
    "sell_cost_pct": sell_cost_list,
    "state_space": state_space,
    "stock_dim": stock_dimensions,
    "tech_indicator_list": INDICATORS,
    "action_space": stock_dimensions,
    "reward_scaling": 1e-4
}
#Setup of the environment with given attributes and data set
e_training_gym = StockTradingEnv(df = train, **env_tweeks)


In [6]:
#Environment for Training

env_train, _ = e_training_gym.get_sb_env()
print(type(env_train))

<class 'stable_baselines3.common.vec_env.dummy_vec_env.DummyVecEnv'>


In [7]:
#Setup Agent

agent = DRLAgent(env = env_train)

#Set if_using_MODEL = true if u want to use other algorithms
if_using_a2c = True
if_using_ddpg = False
if_using_ppo = False
if_using_td3 = False
if_using_sac = False

In [8]:
#Using model A2C
model_a2c =agent.get_model("a2c")

if if_using_a2c:
    #setup new logger
    tmp_path = RESULTS_DIR + '/a2c'
    new_logger_a2c = configure(tmp_path, ["stdout", "csv", "tensorboard"])
    #Set new logger
    model_a2c.set_logger(new_logger_a2c)

#Using model DDPG
model_ddpg =agent.get_model("ddpg")

if if_using_ddpg:
    #setup new logger
    tmp_path = RESULTS_DIR + '/ddpg'
    new_logger_ddpg = configure(tmp_path, ["stdout", "csv", "tensorboard"])
    #Set new logger
    model_ddpg.set_logger(new_logger_ddpg)

#Using model PPO
model_ppo =agent.get_model("ppo")

if if_using_ppo:
    #setup new logger
    tmp_path = RESULTS_DIR + '/ppo'
    new_logger_ppo = configure(tmp_path, ["stdout", "csv", "tensorboard"])
    #Set new logger
    model_ppo.set_logger(new_logger_ppo)

#Using model TD3
##model_td3 =agent.get_model("td3")

#if if_using_td3:
    #setup new logger
    #tmp_path = RESULTS_DIR + '/td3'
    #new_logger_td3 = configure(tmp_path, ["stdout", "csv", "tensorboard"])
    #Set new logger
   # model_td3.set_logger(new_logger_td3)

#Using model SAC
model_sac =agent.get_model("sac")

if if_using_sac:
    #setup new logger
    tmp_path = RESULTS_DIR + '/sac'
    new_logger_sac = configure(tmp_path, ["stdout", "csv", "tensorboard"])
    #Set new logger
    model_sac.set_logger(new_logger_sac)

{'n_steps': 5, 'ent_coef': 0.01, 'learning_rate': 0.0007}
Using cpu device
Logging to results/a2c
{'batch_size': 128, 'buffer_size': 50000, 'learning_rate': 0.001}
Using cpu device
{'n_steps': 2048, 'ent_coef': 0.01, 'learning_rate': 0.00025, 'batch_size': 64}
Using cpu device
{'batch_size': 64, 'buffer_size': 100000, 'learning_rate': 0.0001, 'learning_starts': 100, 'ent_coef': 'auto_0.1'}
Using cpu device


/home/timoross/.local/share/hatch/env/virtual/pro-finrl/G16_L85e/pro-finrl/lib/python3.13/site-packages/stable_baselines3/common/buffers.py:242: UserWarning: This system does not have apparently enough memory to store the complete replay buffer 3.48GB > 2.16GB
  warnings.warn(


In [9]:
#Use loggs to train data. Add aditional blocks for other models if needed
trained_a2c = agent.train_model(
    model = model_a2c,
    tb_log_name = 'a2c',
    total_timesteps = 500
)if if_using_a2c else None

------------------------------------------
| time/                 |                |
|    fps                | 5              |
|    iterations         | 100            |
|    time_elapsed       | 98             |
|    total_timesteps    | 500            |
| train/                |                |
|    entropy_loss       | -619           |
|    explained_variance | 0              |
|    learning_rate      | 0.0007         |
|    n_updates          | 99             |
|    policy_loss        | -0.0259        |
|    reward             | -0.00036992755 |
|    reward_max         | 0.0015087188   |
|    reward_mean        | 0.0002920425   |
|    reward_min         | -0.00087492226 |
|    std                | 1.08           |
|    value_loss         | 2.05e-07       |
------------------------------------------


In [10]:
trained_a2c.save(TRAINED_MODEL_DIR + "/agent_a2c")if if_using_a2c else None